In [ ]:
 # Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import cv2
import shutil
import random
import numpy as np
from pathlib import Path

In [ ]:
# Path folder
input_directory = "/content/drive/MyDrive/mri_otak/Training"
output_directory = "/content/drive/MyDrive/mri_otak/Training_Augmentedv1"
input_augmented_train = "/content/drive/MyDrive/mri_otak/Training_Augmentedv1"
output_split_dir      = "/content/drive/MyDrive/mri_otak/Dataset_Split"

In [ ]:
def resize_and_augment_images(input_directory, output_directory,
                                    target_size=(224, 224)):

    Path(output_directory).mkdir(parents=True, exist_ok=True)

    class_folders = [f for f in os.listdir(input_directory)
                     if os.path.isdir(os.path.join(input_directory, f))]
    print(f"Ditemukan {len(class_folders)} kelas: {class_folders}\n")


    AUGMENTATIONS = {
        "resize" : lambda img: img,
        "rot90"  : lambda img: cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE),
        "rot180" : lambda img: cv2.rotate(img, cv2.ROTATE_180),
        "rot270" : lambda img: cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE),
    }

    grand_total = 0

    for class_name in class_folders:
        input_class_path  = os.path.join(input_directory, class_name)
        output_class_path = os.path.join(output_directory, class_name)
        Path(output_class_path).mkdir(parents=True, exist_ok=True)

        image_files = [f for f in os.listdir(input_class_path)
                       if f.lower().endswith('.jpg')]

        saved = 0
        for img_file in image_files:
            img = cv2.imread(os.path.join(input_class_path, img_file))
            if img is None:
                continue

            img_resized = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
            fname       = os.path.splitext(img_file)[0]

            for aug_name, aug_fn in AUGMENTATIONS.items():
                out_name = f"{fname}_{aug_name}.jpg"
                cv2.imwrite(os.path.join(output_class_path, out_name), aug_fn(img_resized))
                saved += 1

        print(f"{class_name:<15} (asli: {len(image_files):>3}) → {saved} gambar disimpan "
              f"[resize: {len(image_files)}, rot90: {len(image_files)}, "
              f"rot180: {len(image_files)}, rot270: {len(image_files)}]")
        grand_total += saved

    print(f"\n Augmentasi selesai. Total: {grand_total} gambar")
    print(f"Output: {output_directory}")

In [ ]:
# Jalankan
resize_and_augment_images(input_directory, output_directory)

Ditemukan 3 kelas: ['pituitari', 'glioma', 'meningioma']

[selesai] pituitari       (asli: 827) → 3308 gambar disimpan [resize: 827, rot90: 827, rot180: 827, rot270: 827]
[selesai] glioma          (asli: 826) → 3304 gambar disimpan [resize: 826, rot90: 826, rot180: 826, rot270: 826]
[selesai] meningioma      (asli: 822) → 3288 gambar disimpan [resize: 822, rot90: 822, rot180: 822, rot270: 822]

 Augmentasi selesai. Total: 9900 gambar
Output: /content/drive/MyDrive/mri_otak/Training_Augmentedv1


In [ ]:
def split_train_test(input_augmented_train, output_split_dir,
                     train_ratio=0.95, seed=42):

    random.seed(seed)

    output_train = os.path.join(output_split_dir, "train")
    output_test  = os.path.join(output_split_dir, "test")

    class_folders = [f for f in os.listdir(input_augmented_train)
                     if os.path.isdir(os.path.join(input_augmented_train, f))]

    print(f"{'Kelas':<15} {'Total':>7} {'Train (95%)':>12} {'Test (5%)':>10}")
    print("-" * 47)

    grand_train = 0
    grand_test  = 0

    for class_name in sorted(class_folders):
        class_path = os.path.join(input_augmented_train, class_name)
        files = [os.path.join(class_path, f)
                 for f in os.listdir(class_path)
                 if f.lower().endswith('.jpg')]

        random.shuffle(files)

        n_train = round(len(files) * train_ratio)
        n_test  = len(files) - n_train

        train_files = files[:n_train]
        test_files  = files[n_train:]

        # Buat folder output
        Path(os.path.join(output_train, class_name)).mkdir(parents=True, exist_ok=True)
        Path(os.path.join(output_test,  class_name)).mkdir(parents=True, exist_ok=True)

        for f in train_files:
            shutil.copy(f, os.path.join(output_train, class_name, os.path.basename(f)))

        for f in test_files:
            shutil.copy(f, os.path.join(output_test, class_name, os.path.basename(f)))

        print(f"{class_name:<15} {len(files):>7} {n_train:>12} {n_test:>10}")
        grand_train += n_train
        grand_test  += n_test

    print("-" * 47)
    print(f"{'TOTAL':<15} {grand_train+grand_test:>7} {grand_train:>12} {grand_test:>10}")
    print(f"\n Split selesai!")
    print(f"Train : {output_train}")
    print(f"Test  : {output_test}")

In [ ]:
# Jalankan
split_train_test(input_augmented_train, output_split_dir)

Kelas             Total  Train (95%)  Test (5%)
-----------------------------------------------
glioma             3304         3139        165
meningioma         3288         3124        164
pituitari          3308         3143        165
-----------------------------------------------
TOTAL              9900         9406        494

 Split selesai!
Train : /content/drive/MyDrive/mri_otak/Dataset_Split/train
Test  : /content/drive/MyDrive/mri_otak/Dataset_Split/test
